<a href="https://colab.research.google.com/github/freida20git/bird-detection-tracking/blob/main/track%26count.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 41.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
# get the files needed
!gdown "https://drive.google.com/uc?id=1ddKOxyxFS8jUNBSknbaumvI16tNMDqFR"
!gdown "https://drive.google.com/uc?id=1S9jEgd9m6O6O9srOapjyFG0tUU0IzfXM"
!gdown 1XGeddfek_Q21YUW5OwZgS9QUR5c6NY-S -O similar_birds_sky.mp4
!gdown 1nQHp1ze49jPNC0lgUY7yMV5NOBBwTC3i -O oclusion_birds.mp4
!gdown 1VsGCH9IXCSkI75LbMgIA63gv2yMTu82a -O sunset.mp4
!gdown 1fSmfgg1vQLNtKAKbdy71LWpK6D2MY5rL -O birds_beach.mp4

In [ ]:
!gdown 'https://drive.google.com/uc?id=16XVu6YHRthELaPRXiG9DRPkbtPOM7r49'

Downloading...
From: https://drive.google.com/uc?id=16XVu6YHRthELaPRXiG9DRPkbtPOM7r49
To: /content/10489468-hd_1080_1920_30fps.mp4
100% 5.40M/5.40M [00:00<00:00, 19.1MB/s]


In [ ]:
!gdown 'https://drive.google.com/uc?id=1x-A9WOyZtZrOqlgM-EXg5g4ek6YgmCYO'

Downloading...
From: https://drive.google.com/uc?id=1x-A9WOyZtZrOqlgM-EXg5g4ek6YgmCYO
To: /content/bestbirdsonly.pt
100% 5.47M/5.47M [00:00<00:00, 146MB/s]


In [ ]:
# Install yt-dlp
!pip install yt-dlp

# Download a YouTube video
!yt-dlp 'https://www.youtube.com/watch?v=KLezgwLA_94'

In [ ]:
!yolo task=detect mode=track model=bestbirdsonly.pt source='Flying Birds 4K Video - Free HD Stock Footage - No Copyright - Bird Animals Sky [KLezgwLA_94].webm' show=True save=True

In [ ]:
import cv2
import numpy as np
import json
from ultralytics import YOLO
import matplotlib.cm as cm # Import colormaps
import matplotlib.colors as mcolors # Import color handling
from collections import deque # Use deque for efficient storage of trail points
import random # Import random for initial color generation


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# common functions for tracking counts
def bbox_center(bbox):
    """Calculate the center point of a bounding box."""
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, (y1 + y2) / 2)

def initialize_video_objects(video_path):
    """Initialize video capture and writer objects."""
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    return cap, width, height, fps

def create_video_writer(output_path, width, height, fps):
    """Create and return a video writer object."""
    return cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

def get_track_color(track_colors, track_id, colormap):
    """Get or assign a color for a track ID."""
    if track_id not in track_colors:
        color_index = track_id % 20
        rgb_color = colormap(color_index)[:3]
        bgr_color = (int(rgb_color[2] * 255), int(rgb_color[1] * 255), int(rgb_color[0] * 255))
        track_colors[track_id] = bgr_color
    return track_colors[track_id]

def update_track_history(track_history, track_id, current_center, trail_length=50):
    """Update the track history with new center points."""
    if track_id not in track_history or track_history[track_id] is None:
        track_history[track_id] = deque(maxlen=trail_length)
    track_history[track_id].append(current_center)
    return track_history

def match_detection_to_track(detection_map, current_center, max_distance_sq=400):
    """Match a track to the closest detection for class information."""
    matched_det = None
    min_dist_sq = float('inf')

    for det in detection_map:
        if not det.get("used", False):
            det_center = bbox_center(det["bbox"])
            dist_sq = (current_center[0] - det_center[0])**2 + (current_center[1] - det_center[1])**2
            if dist_sq < max_distance_sq and dist_sq < min_dist_sq:
                min_dist_sq = dist_sq
                matched_det = det
                matched_det["used"] = True

    return matched_det

def draw_tracking_elements(frame, tracks, track_history, track_colors, detection_map, model, colormap):
    """Draw all tracking elements on the frame."""
    current_tracked_ids = set()

    for track in tracks:
        if len(track) >= 5:  # Check if track has [x1, y1, x2, y2, id, ...]
            x1, y1, x2, y2, track_id = track[:5]
            track_id = int(track_id)
            current_tracked_ids.add(track_id)

            current_color = get_track_color(track_colors, track_id, colormap)
            current_center = (int((x1 + x2) / 2), int((y1 + y2) / 2))
            track_history = update_track_history(track_history, track_id, current_center)

            matched_det = match_detection_to_track(detection_map, current_center)

            # Draw bounding box
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), current_color, 2)

            # Draw class name if available
            if matched_det and hasattr(model, 'names'):
                class_id = matched_det.get("class_id")
                if class_id is not None and class_id < len(model.names):
                    class_name = model.names[class_id]
                    text = f'{class_name}'
                    cv2.putText(frame, text, (int(x1), int(y1) - 10),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.6, current_color, 2)

    # Draw trails
    for track_id, centers in track_history.items():
        if track_id in track_colors and centers:
            trail_color = track_colors[track_id]
            for i in range(1, len(centers)):
                cv2.line(frame, centers[i-1], centers[i], trail_color, 2)

    # Add tracking count text
    cv2.putText(frame, f"{len(current_tracked_ids)} Birds Tracked", (50, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2, cv2.LINE_AA)

    return frame, current_tracked_ids

### SORT: cloned from https://raw.githubusercontent.com/abewley/sort/master/sort.py with minor modifications

In [ ]:
!pip install ultralytics filterpy opencv-python

In [ ]:
!wget https://raw.githubusercontent.com/freida20git/bird-detection-tracking/main/sort.py

--2025-06-19 19:21:17--  https://raw.githubusercontent.com/freida20git/bird-detection-tracking/main/sort.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11639 (11K) [text/plain]
Saving to: ‘sort.py’

sort.py             100%[===================>]  11.37K  --.-KB/s    in 0.003s  

2025-06-19 19:21:17 (4.18 MB/s) - ‘sort.py’ saved [11639/11639]



In [ ]:
from sort import *
from ultralytics import YOLO

def initialize_sort_tracker():
    """Initialize and return the SORT tracker."""
    KalmanBoxTracker.count = 0
    return Sort(max_age=30, min_hits=3, iou_threshold=0.1)

def process_sort_detections(results):
    """Process YOLO detection results for SORT."""
    detections = []
    det_info = []

    for i, box in enumerate(results[0].boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        detections.append([x1, y1, x2, y2, conf])
        det_info.append({
            "index": i,
            "bbox": (x1, y1, x2, y2),
            "conf": conf,
            "class_id": cls_id,
            "used": False
        })

    return np.array(detections) if detections else np.empty((0, 5)), det_info

def main_sort():
    """Main function for SORT tracking."""
    cap, width, height, fps = initialize_video_objects('/content/10489468-hd_1080_1920_30fps.mp4')
    out = create_video_writer('output_sort_trails.mp4', width, height, fps)
    model = YOLO('bestbirdsonly.pt')
    tracker = initialize_sort_tracker()

    track_history = {}
    track_colors = {}
    colormap = cm.get_cmap('tab10', 20)
    unique_ids = set()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model.predict(frame, conf=0.3, classes=0, iou=0.4, verbose=False)
        dets, det_info = process_sort_detections(results)
        tracks = tracker.update(dets)

        frame, current_ids = draw_tracking_elements(
            frame, tracks, track_history, track_colors, det_info, model, colormap
        )
        unique_ids.update(current_ids)
        out.write(frame)

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"Total number of birds detected: {len(unique_ids)}")

if __name__ == "__main__":
    main_sort()

/tmp/ipython-input-25-40271981.py:38: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap = cm.get_cmap('tab10', 20)


Total number of birds detected: 14


In [ ]:
!rm "/content/result_compressed.mp4"

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

# Input video path
save_path = '/content/output_sort_trails.mp4'

compressed_path = "/content/result_compressed.mp4"

os.system(f"ffmpeg -i {save_path} -vcodec libx264 {compressed_path}")

# Show video
mp4 = open(compressed_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

### OcSORT:


In [ ]:
!git clone https://github.com/noahcao/OC_SORT.git
!cd OC_SORT
!pip3 install -r /content/OC_SORT/requirements.txt
!python3 setup.py develop

In [ ]:
!apt-get install -y build-essential
!pip install cython
!pip install lap
!pip install filterpy

In [ ]:
# make sure the ocsort object file exists:
!ls /content/OC_SORT/trackers/ocsort_tracker

association.py	kalmanfilter.py  ocsort.py


In [ ]:
import sys
sys.path.append('/content/OC_SORT') # Add the parent directory to sys.path

from trackers.ocsort_tracker.ocsort import OCSort

tracker = OCSort(det_thresh=0.5)
print("OCSort loaded successfully!")

OCSort loaded successfully!


In [35]:
from trackers.ocsort_tracker.ocsort import OCSort

def initialize_ocsort_tracker():
    """Initialize and return the OC-SORT tracker."""
    return OCSort(
        det_thresh=0.4,
        max_age=90,
        min_hits=2,
        iou_threshold=0.3,
        delta_t=3,
        inertia=0.3,
        asso_func="ciou"
    )

def process_ocsort_detections(results):
    """Process YOLO detection results for OC-SORT with class IDs."""
    detections = results[0].boxes
    if detections is not None and len(detections) > 0:
        boxes = detections.xyxy.cpu().numpy()
        scores = detections.conf.cpu().numpy()
        class_ids = detections.cls.cpu().numpy()  # Extract class IDs
        dets = np.hstack((boxes, scores.reshape(-1, 1)))

        detection_map = []
        for i in range(len(dets)):
            detection_map.append({
                "bbox": dets[i][:4],
                "confidence": float(dets[i][4]),
                "class_id": int(class_ids[i]),  # Add class ID to detection
                "used": False
            })
        return dets, detection_map
    return np.empty((0, 5)), []

def main_ocsort():
    """Main function for OC-SORT tracking."""
    cap, width, height, fps = initialize_video_objects('/content/10489468-hd_1080_1920_30fps.mp4')
    out = create_video_writer("output_ocsort_trails.mp4", width, height, fps)
    model = YOLO('bestbirdsonly.pt')
    tracker = initialize_ocsort_tracker()

    track_history = {}
    track_colors = {}
    colormap = cm.get_cmap('tab10', 20)
    unique_ids = set()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model.predict(frame, conf=0.3, iou=0.4, verbose=False)
        dets, detection_map = process_ocsort_detections(results)
        tracks = tracker.update(dets, frame.shape[:2], frame.shape[:2])

        frame, current_ids = draw_tracking_elements(
            frame, tracks, track_history, track_colors, detection_map, model, colormap
        )
        unique_ids.update(current_ids)
        out.write(frame)

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"Total number of birds detected: {len(unique_ids)}")

if __name__ == "__main__":
    main_ocsort()

/tmp/ipython-input-35-2744050949.py:44: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap = cm.get_cmap('tab10', 20)


Total number of birds detected: 8


# video output:

In [36]:
!rm "/content/result_compressed.mp4"

In [37]:
from IPython.display import HTML
from base64 import b64encode
import os

# Input video path
save_path = '/content/output_ocsort_trails.mp4'

compressed_path = "/content/result_compressed.mp4"

os.system(f"ffmpeg -i {save_path} -vcodec libx264 {compressed_path}")

# Show video
mp4 = open(compressed_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)